In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#load the dataset
df_raw = pd.read_csv("../data/superstore.csv")

print("Original Dataset Shape:", df_raw.shape)

df_raw.head()


In [ ]:
#clean the dataset
# Keep only actual sales records
df = df_raw[
    pd.to_datetime(
        df_raw["Order Date"],
        format="%m/%d/%Y",
        errors="coerce"
    ).notna()
].copy()

# Convert date columns
df["Order Date"] = pd.to_datetime(
    df["Order Date"],
    format="%m/%d/%Y"
)

df["Ship Date"] = pd.to_datetime(
    df["Ship Date"],
    format="%m/%d/%Y"
)

# Convert numerical columns
numeric_columns = [
    "Sales", "Profit", "Quantity", "Discount"
]

df[numeric_columns] = df[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

print("Cleaned Dataset Shape:", df.shape)

In [ ]:
#verify data quality
print("Missing Values:")
print(df.isnull().sum())

print("\nDuplicate Records:")
print(df.duplicated().sum())

print("\nDataset Information:")
df.info()

In [ ]:
# Overall business performance

total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order ID"].nunique()
total_customers = df["Customer ID"].nunique()
total_products = df["Product ID"].nunique()

print("SALES PERFORMANCE SUMMARY")
print("-" * 35)
print(f"Total Sales:     ${total_sales:,.2f}")
print(f"Total Profit:    ${total_profit:,.2f}")
print(f"Total Orders:    {total_orders:,}")
print(f"Total Customers: {total_customers:,}")
print(f"Unique Products: {total_products:,}")

profit_margin = (total_profit / total_sales) * 100
print(f"Profit Margin:   {profit_margin:.2f}%")

In [ ]:
# Monthly sales analysis

df["Year-Month"] = df["Order Date"].dt.to_period("M")

monthly_sales = (
    df.groupby("Year-Month")["Sales"]
    .sum()
    .reset_index()
)

monthly_sales["Year-Month"] = monthly_sales["Year-Month"].astype(str)

print(monthly_sales.head(10))

plt.figure(figsize=(14, 6))

sns.lineplot(
    data=monthly_sales,
    x="Year-Month",
    y="Sales",
    marker="o"
)

plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
#Analyze sales and profit by category

category_performance = (
    df.groupby("Category")
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Total_Quantity=("Quantity", "sum")
    )
    .reset_index()
)

category_performance["Profit_Margin"] = (
    category_performance["Total_Profit"]
    / category_performance["Total_Sales"]
) * 100

category_performance

plt.figure(figsize=(9, 5))

sns.barplot(
    data=category_performance,
    x="Category",
    y="Total_Sales"
)

plt.title("Sales by Product Category")
plt.xlabel("Category")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.show()

In [ ]:
#Find the top 10 products

top_products = (
    df.groupby("Product Name")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

top_products

plt.figure(figsize=(12, 6))

sns.barplot(
    data=top_products,
    x="Sales",
    y="Product Name"
)

plt.title("Top 10 Products by Sales")
plt.xlabel("Total Sales")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

In [ ]:
#Analyze regional performance

region_performance = (
    df.groupby("Region")
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum")
    )
    .reset_index()
)

region_performance

plt.figure(figsize=(9, 5))

sns.barplot(
    data=region_performance,
    x="Region",
    y="Total_Profit"
)

plt.title("Profit by Region")
plt.xlabel("Region")
plt.ylabel("Total Profit")
plt.tight_layout()
plt.show()

In [ ]:
# Analyze average discount and total profit by product

discount_analysis = (
    df.groupby("Product Name")
    .agg(
        Average_Discount=("Discount", "mean"),
        Total_Profit=("Profit", "sum"),
        Total_Sales=("Sales", "sum")
    )
    .reset_index()
)

discount_analysis = discount_analysis.sort_values(
    by="Average_Discount",
    ascending=False
)

discount_analysis.head(10)

In [ ]:
# Identify products with high discounts and negative profit

loss_products = discount_analysis[
    (discount_analysis["Average_Discount"] > 0.30) &
    (discount_analysis["Total_Profit"] < 0)
].sort_values(by="Total_Profit")

loss_products.head(15)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=loss_products.head(10),
    x="Total_Profit",
    y="Product Name"
)

plt.title("Top 10 Loss-Making Products with High Discounts")
plt.xlabel("Total Profit")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

In [ ]:
#Analyze profit margin by category
category_margin = (
    df.groupby("Category")
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum")
    )
    .reset_index()
)

category_margin["Profit_Margin"] = (
    category_margin["Total_Profit"] /
    category_margin["Total_Sales"]
) * 100

category_margin

plt.figure(figsize=(9, 5))

sns.barplot(
    data=category_margin,
    x="Category",
    y="Profit_Margin"
)

plt.title("Profit Margin by Category")
plt.xlabel("Category")
plt.ylabel("Profit Margin (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Final business insights

best_category = category_margin.loc[
    category_margin["Total_Profit"].idxmax()
]

best_region = region_performance.loc[
    region_performance["Total_Profit"].idxmax()
]

best_product = top_products.loc[
    top_products["Sales"].idxmax()
]

print("FINAL BUSINESS INSIGHTS")
print("=" * 40)

print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")

print(
    f"Most Profitable Category: "
    f"{best_category['Category']}"
)

print(
    f"Most Profitable Region: "
    f"{best_region['Region']}"
)

print(
    f"Top Selling Product: "
    f"{best_product['Product Name']}"
)

print(f"High Discount / Loss Products: {len(loss_products)}")

In [31]:
import os

os.makedirs("../reports", exist_ok=True)

monthly_sales.to_csv("../reports/monthly_sales.csv", index=False)
category_performance.to_csv("../reports/category_performance.csv", index=False)
top_products.to_csv("../reports/top_products.csv", index=False)
region_performance.to_csv("../reports/region_performance.csv", index=False)
loss_products.to_csv("../reports/loss_products.csv", index=False)
category_margin.to_csv("../reports/category_margin.csv", index=False)

print("All analysis reports saved successfully!")

All analysis reports saved successfully!
